In [0]:
# ============================================================
# SILVER LAYER (Part 1) — Event Dimension
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

spark.sql("USE CATALOG iran_israel_capstone_project")
spark.sql("USE SCHEMA bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

In [0]:
# ============================================================
# STEP 4 — BUILD silver.event_dim
# ============================================================
event_dim = (
    spark.table("bronze.events")
    .select(
        F.col("event_date").cast(DateType()).alias("event_date"),
        F.col("event_id"),
        F.col("event_type"),
        F.col("severity"),
        F.col("crude_risk"),
        F.col("description"),
        F.col("source_url"),
        F.col("t_plus_1_expected"),
    )
    .filter(F.col("event_date").isNotNull())
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("event_type").isNotNull())
    .filter(F.col("severity").isNotNull())
    .dropDuplicates(["event_id"])
    .withColumn("silver_timestamp", F.current_timestamp())
)

In [0]:
# Validate: must have >= 12 events
event_count = event_dim.count()
assert event_count >= 12, f"❌ event_dim has only {event_count} rows — need >= 12"
print(f"✅ event_dim row count: {event_count}")

(
    event_dim.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.event_dim")
)
print("✅ silver.event_dim written")